In [1]:
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.messages import TextMessage
import asyncio

# Setup local brain (Llama 3.2 via Ollama)
model_client = OpenAIChatCompletionClient(
    model="llama3.2",
    api_key="placeholder",
    base_url="http://localhost:11434/v1",
    model_info={
        "vision": False, "function_calling": True, "json_output": True,
        "family": "llama3", "structured_output": True
    },
)

##The tool Defining the Action 
The web search tool is an async function essential for M1 mac to handle background task without freezing 

In [2]:
async def web_search(query: str) -> str:
    """Find information on the web regarding agricultural technical specs."""
    # In a real scenario, this would call a search API.
    # For this test, it returns a deterministic string.
    return "The Labrador Retriever or simply Labrador is a British breed of retriever gun dog."

The Orchestration: Creating the Agent

In [3]:
agent = AssistantAgent(
    name="assistant",
    model_client=model_client,
    tools=[web_search],
    system_message="Use Tools to solve tasks accurately.",
    description="An agent which uses tools to help solve tasks."
)

In [4]:
async def main():
    print("--- Tool Use Execution Started ---")
    result = await agent.run(task="Find information about Labrador Retriever")
    
    # We print the last message content to see the final verified answer
    print(f"\nFinal Answer: {result.messages[-1].content}")

if __name__ == "__main__":
    await main() # Run this in your Jupyter notebook cell

--- Tool Use Execution Started ---

Final Answer: The Labrador Retriever or simply Labrador is a British breed of retriever gun dog.


In [5]:
from autogen_agentchat.ui import Console
from autogen_core import CancellationToken

async def assistant_run_stream() -> None:
    # This 'Console' call wraps the agent's stream
    await Console(
        agent.on_messages_stream(
            messages=[TextMessage(content='Find info about Labrador Retriever via the tool', source='User')],
            cancellation_token=CancellationToken()
        ),
        output_stats=True # This shows token usage and duration—critical for your M1 Mac monitoring
    )

# Execution call
await assistant_run_stream()

---------- ToolCallRequestEvent (assistant) ----------
[FunctionCall(id='call_ngq4eqa3', arguments='{"query":"Labrador Retriever breed information"}', name='web_search')]
[Prompt tokens: 228, Completion tokens: 23]
---------- ToolCallExecutionEvent (assistant) ----------
[FunctionExecutionResult(content='The Labrador Retriever or simply Labrador is a British breed of retriever gun dog.', name='web_search', call_id='call_ngq4eqa3', is_error=False)]
---------- assistant ----------
The Labrador Retriever or simply Labrador is a British breed of retriever gun dog.
---------- Summary ----------
Number of inner messages: 2
Total prompt tokens: 228
Total completion tokens: 23
Duration: 6.32 seconds
